# Building the RAG dataset for Object Classification

In [ ]:
import os
import io
import time

import pandas as pd
import requests

## Preparing the vocabularies

### AAT (from SPARQL endpoint)

The code below reads vocab entries from Getty's AAT, through their SPARQL endpoint, and creates a dataset from the URI, label and defintion.

In [ ]:
# The URL for the Getty SPARQL endpoint
SPARQL_ENDPOINT = "http://vocab.getty.edu/sparql"

# The SPARQL query to retrieve URI, label, and description for English terms
sparql_query = """
PREFIX gvp: <http://vocab.getty.edu/ontology#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX aat: <http://vocab.getty.edu/aat/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?term ?definition ?uri
WHERE {
  ?uri a gvp:Concept;
       gvp:prefLabelGVP [
         gvp:term ?rterm;
         dcterms:language aat:300388277
       ];
       skos:scopeNote [
         dcterms:language aat:300388277;
         rdf:value ?rdefinition
       ].
  BIND (STR(?rterm)  AS ?term)
  BIND (STR(?rdefinition)  AS ?definition)
}
"""

def get_aat_concepts(query):
    """
    Queries the Getty SPARQL endpoint and returns a Pandas DataFrame.
    """
    headers = {
        # We ask for CSV format as it's easy to parse with Pandas
        "Accept": "text/csv"
    }
    params = {
        "query": query
    }
    
    print("Querying Getty AAT SPARQL endpoint... This may take several minutes.")
    try:
        response = requests.get(SPARQL_ENDPOINT, params=params, headers=headers)
        response.raise_for_status()  # Raise an exception for bad status codes
        
        print("Query successful. Parsing results into DataFrame...")
        # Use io.StringIO to treat the CSV response text as a file
        csv_data = io.StringIO(response.text)
        df = pd.read_csv(csv_data)
        
        # Add the source vocabulary column
        df['source_vocabulary'] = 'AAT'
        
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None


In [ ]:

aat_df = get_aat_concepts(sparql_query)
if aat_df is not None:
    print(f"\nSuccessfully retrieved {len(aat_df)} terms from the Getty AAT.")
    # You can now save this directly to your own CSV for later use
    # aat_df.to_csv('aat_vocabulary.csv', index=False)
    # print("Saved data to aat_vocabulary.csv")

In [ ]:
aat_df

### PBCore (Excel)

PBCore's vocabularies are released in a series of Excel spreadsheets: these are re-organised as for AAT.

In [ ]:
!ls vocabularies/pbcore/

In [ ]:
pbcore_dir = "vocabularies/pbcore/"

pbcore_assettype = pd.read_excel(
    os.path.join(pbcore_dir, "pbcore_pbcoreassettype_vocabulary.xlsx"))

# pbcore_instantiations are 3 files starting with pbcore_pbcoreinstantiation
# Let's read them all and concatenate them into a single DataFrame
pbcore_instantiations_dfs = []
for f in os.listdir(pbcore_dir):
    if f.startswith("pbcore_instantiation") and f.endswith(".xlsx"):
        print(f"Reading {f}...")
        file_path = os.path.join(pbcore_dir, f)
        df = pd.read_excel(file_path)
        pbcore_instantiations_dfs.append(df)

# Concatenate all DataFrames into one
pbcore_instantiations = pd.concat(pbcore_instantiations_dfs, ignore_index=True)

In [ ]:
pbcore_assettype.rename(columns={
    'Term': 'term',
    'Definition': 'definition'
}, inplace=True)
# Add source vocabulary column
pbcore_assettype['source_vocabulary'] = 'PBCore-AssetType'
print(f"Number of terms in PBCore AssetType: {len(pbcore_assettype)}")

# same for pbcore_instantiations
pbcore_instantiations.rename(columns={
    'Term': 'term',
    'Definition': 'definition'
}, inplace=True)
# Add source vocabulary column
pbcore_instantiations['source_vocabulary'] = 'PBCore-Instantiation'
print(f"Number of terms in PBCore Instantiation: {len(pbcore_instantiations)}")

In [ ]:
# Now merge the two DataFrames into a single one
pbcore_vocabulary = pd.concat([pbcore_assettype, pbcore_instantiations], ignore_index=True)
# Make sure that we have 14 + 79 terms there
assert len(pbcore_vocabulary) == len(pbcore_instantiations) + len(pbcore_assettype)
pbcore_vocabulary

In [ ]:
# and save it
pbcore_vocabulary.to_csv(
    os.path.join(pbcore_dir, "pbcore_concepts.csv"), index=False)

### MIMO (Turtle)

MIMO is provided in Turtle from the controlled vocabularies of the DOREMUS project (https://data.doremus.org/vocabularies/). These include URI and multi-lingual labels, but no definition (which is also not provided by MIMO's thesaurus); as well as skos similar matches to DBPedia (instead, MIMO provide equivalent ones in their Turtle). The workflow below: (1) filters out the English entries in the vocabulary and retain their URI and label; (2) queries MIMO through their API to retrieve the Turtle data of a certain instrument; (3) if an exact match to a Wikidata (instrument) entity is found, the description is retrieve from (4) a SPARQL query to DBPedia. A dataset is constructed accordingly. 

In [ ]:
import pandas as pd
import requests
import time
import re

from rdflib import Graph
from tqdm import tqdm

# --- 1. CONFIGURATION ---
MIMO_TURTLE_FILE = "vocabularies/mimo/mimo_doremus.ttl"
MIMO_API_BASE_URL = "https://vocabulary.mimo-international.com/rest/v1/InstrumentsKeywords/data"
DBPEDIA_SPARQL_ENDPOINT = "https://dbpedia.org/sparql"
WIKIDATA_SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"


mimo_graph = Graph()
print(f"Loading MIMO vocabulary from {MIMO_TURTLE_FILE}...")
mimo_graph.parse(MIMO_TURTLE_FILE, format='turtle')

In [ ]:

def get_dbpedia_link_from_mimo_api(instrument_uri: str) -> str | None:
    """
    Queries the MIMO API for a specific instrument to find its DBpedia link.
    """
    params = {"uri": instrument_uri, "format": "text/turtle"}
    try:
        response = requests.get(MIMO_API_BASE_URL, params=params, timeout=10)
        response.raise_for_status()

        # Parse the small turtle response in memory
        g_single = Graph()
        g_single.parse(data=response.text, format="turtle")
        # Query this small graph for the DBpedia link
        query = """
            PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
            SELECT ?dbPediaLink WHERE {
                ?s skos:exactMatch ?dbPediaLink .
                FILTER(CONTAINS(STR(?dbPediaLink), "dbpedia.org"))
            }
        """
        results = g_single.query(query)
        for row in results:
            return str(row.dbPediaLink) # Return the first one found
            
    except requests.exceptions.RequestException as e:
        print(f"  - MIMO API Error for {instrument_uri}: {e}")
    return None


def get_dbpedia_description(dbpedia_uri: str) -> str | None:
    """
    Queries the DBpedia SPARQL endpoint to get the English comment (description),
    using the correct Accept header to avoid a 406 error.
    """
    # Replace /page/ with /resource/ to get the correct data URI
    if "/resource/" not in dbpedia_uri:
        if "/page/" not in dbpedia_uri:
            print(f"  - Warning: Expected '/page/' URI but got: {dbpedia_uri}")
            return None
        resource_uri = dbpedia_uri.replace("/page/", "/resource/")
    # Replace https with http if needed
    if resource_uri.startswith("https://"):
        resource_uri = resource_uri.replace("https://", "http://")

    sparql_query = f"""
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?d WHERE {{
            <{resource_uri}> rdfs:comment ?d .
            FILTER(LANG(?d) = 'en')
        }} LIMIT 1
    """
    # Headers and params
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    
    try:
        response = requests.get(
            DBPEDIA_SPARQL_ENDPOINT,
            params=params, headers=headers, timeout=10)
        response.raise_for_status()
        results = response.json()["results"]["bindings"]
        if results:  # careful, this may be empty
            return results[0]["d"]["value"]
    except Exception as e:
        print(f"  - DBpedia Error for {resource_uri}: {e}")
        
    return None


def get_wikidata_info(mimo_uri: str) -> tuple[str, str] | None:
    """
    Queries Wikidata to find an item matching the MIMO Instrument ID (P3763).
    Returns a tuple of (description, wikidata_uri) if found.
    """
    mimo_id_match = re.search(r'(\d+)$', mimo_uri)
    if not mimo_id_match: return None
    mimo_id = mimo_id_match.group(1)

    sparql_query = f"""
        SELECT ?item ?itemLabel ?itemDescription WHERE {{
            ?item wdt:P3763 "{mimo_id}" .
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
        }} LIMIT 1
    """
    headers = {"Accept": "application/json", "User-Agent": "MyCoolMuseumProject/1.0"}
    params = {"query": sparql_query}

    try:
        response = requests.get(
            WIKIDATA_SPARQL_ENDPOINT, params=params, headers=headers, timeout=15)
        response.raise_for_status()
        results = response.json()["results"]["bindings"]
        if results:
            description = results[0]["itemDescription"]["value"]
            item_uri = results[0]["item"]["value"]
            return description, item_uri
    except Exception: pass
    return None

# Testing the wikidata function with the URI of Electric Guitar (http://www.mimo-db.eu/InstrumentsKeywords/3236)
# mimo_uri = "http://www.mimo-db.eu/InstrumentsKeywords/3236"
# wikidata_info = get_wikidata_info(mimo_uri)
# wikidata_info

In [ ]:
# This query now ONLY gets the URI and English label from the local Turtle file
master_list_query = """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?uri ?label WHERE {
        ?uri skos:prefLabel ?label .
        FILTER(LANG(?label) = "en")
    }
"""
instrument_list = list(mimo_graph.query(master_list_query))
print(f"Found {len(instrument_list)} English terms to process.")

In [ ]:
enriched_data = []

for uri, label in tqdm(instrument_list, desc="MIMO Description enrichment"):
    instrument_uri = str(uri)
    instrument_label = str(label)
    description = ""
    description_source = ""
    dbpedia_ref = None
    wikidata_ref = None

    time.sleep(0.1) # let's be kind to the APIs

    # STAGE 1: Try DBpedia (following the MIMO API's skos:exactMatch)
    db_pedia_link = get_dbpedia_link_from_mimo_api(instrument_uri)
    if db_pedia_link:
        print(f"  - Found DBpedia link for {instrument_label}: {db_pedia_link}")
        dbpedia_ref = db_pedia_link
        desc_from_dbp = get_dbpedia_description(db_pedia_link)
        if desc_from_dbp:
            description = desc_from_dbp
            description_source = "dbpedia"

    # STAGE 2: If DBpedia failed, try Wikidata
    if not description:
        print(f"  - No DBpedia description found for {instrument_label}, trying Wikidata...")
        wikidata_info = get_wikidata_info(instrument_uri)
        if wikidata_info:
            print(f"  - Found Wikidata info for {instrument_label}: {wikidata_info[1]}")
            wikidata_ref = wikidata_info[1]
            description = wikidata_info[0]
            description_source = "wikidata"

    enriched_data.append({
        "term": instrument_label,
        "definition": description,
        "uri": instrument_uri,
        "source_vocabulary": "MIMO",
        "definition_source": description_source,
        "dbpedia_ref": dbpedia_ref,
        "wikidata_ref": wikidata_ref,
    })

In [ ]:
# --- 4. CREATE FINAL DATAFRAME ---
mimo_df = pd.DataFrame(enriched_data)
descriptions_found = mimo_df[mimo_df['definition'] != ''].shape[0]
print(f"\nSuccessfully retrieved {descriptions_found} descriptions.")
mimo_df.to_csv('vocabularies/mimo_concepts_ext.csv', index=False)

In [ ]:
# Tell me how many terms were enriched from DBpedia and from Wikidata   
dbpedia_count = mimo_df[mimo_df['definition_source'] == 'dbpedia'].shape[0]
wikidata_count = mimo_df[mimo_df['definition_source'] == 'wikidata'].shape[0]
print(f"Enriched {dbpedia_count} terms from DBpedia and {wikidata_count} terms from Wikidata.")

### Europeana Fashion vocabulary (RDF)

In [ ]:
import pandas as pd
from rdflib import Graph

EUROPEANA_FASHION_RDF_FILE = 'vocabularies/fashion/fashion.rdf'

sparql_query = """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?uri ?term ?description
    WHERE {
        ?uri rdf:type skos:Concept .
        ?uri skos:prefLabel ?term .
        FILTER(LANG(?term) = "en") .

        OPTIONAL {
            ?uri skos:scopeNote ?description .
            FILTER(LANG(?description) = "en")
        }
    }
"""

In [ ]:
print(f"Parsing Europeana Fashion vocab: {EUROPEANA_FASHION_RDF_FILE}")
g = Graph()

try:
    # The format for RDF/XML files is 'xml'
    g.parse(source=EUROPEANA_FASHION_RDF_FILE, format='xml')
    print("Parsing complete. Extracting vocabularity dataset...")
    results = g.query(sparql_query)
    print(f"Found {len(results)} English terms.")

    # --- 4. Load Results into a Pandas DataFrame ---
    data = []
    for row in tqdm(results):
        data.append({
            "term": str(row.term),
            "definition": str(row.description) if row.description else "",
            "uri": str(row.uri),
        })

    fashion_df = pd.DataFrame(data)
    fashion_df['source_vocabulary'] = 'Europeana'
    fashion_df.to_csv('vocabularies/fashion_concepts.csv', index=False)

except Exception as e:
    print(f"An error occurred during parsing: {e}")

## Create the RAG from embeddings

### Collate and save

In [ ]:
import pandas as pd

vocab_files = [
    'vocabularies/aat/aat_concepts.csv',
    'vocabularies/mimo/mimo_concepts.csv',
    'vocabularies/fashion/fashion_concepts.csv',
    'vocabularies/pbcore/pbcore_concepts.csv'
]

df_list = []
for f in vocab_files:
    print(f"Loading vocabulary from {f}...")
    vocab_df = pd.read_csv(f)
    if "aat_concepts" in f:
        vocab_df["source_vocabulary"] = "AAT"
    vocab_df = vocab_df[['term', 'definition', 'uri', 'source_vocabulary']]
    df_list.append(vocab_df)

all_vocabs_df = pd.concat(df_list, ignore_index=True)
# Ensure essential columns exist and handle missing descriptions
all_vocabs_df['definition'] = all_vocabs_df['definition'].fillna('')

# is there any record with empty or nan uri?
assert (all_vocabs_df['uri'] is None or all_vocabs_df['uri'].isna()).sum() == 0,\
"There are records with empty or NaN URIs!"
# is there any record with empty or nan label?
assert (all_vocabs_df['term'] is None or all_vocabs_df['term'].isna()).sum() == 0,\
"There are records with empty or NaN labels!"

# Replace "Europeana" with "Fashion" in the source vocabulary
all_vocabs_df['source_vocabulary'].replace('Europeana', 'Fashion', inplace=True)

# all_vocabs_df.dropna(subset=['uri', 'label'], inplace=True)
print(f"\nLoaded a total of {len(all_vocabs_df)} terms.")
# Count the number of terms per each vocabulary
vocab_counts = all_vocabs_df['source_vocabulary'].value_counts()
print("Number of terms per vocabulary:")
print(vocab_counts)

In [ ]:
all_vocabs_df.to_csv('vocabularies/all_concepts.csv', index=False)
print("Saved all vocabulary terms to 'vocabularies/all_concepts.csv'.")

### Example of RAG search

In [ ]:
import os
import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [ ]:
# load the concept vocabulary
vocab_df = pd.read_csv('vocabularies/all_concepts.csv')


In [ ]:
index = faiss.read_index("vocabularies/vectordb/all_concepts_min.index")
model = SentenceTransformer('all-MiniLM-L6-v2') # Must use the same model!


In [ ]:
description = "The Blues Brother Gibson ES-335"

query_vector = model.encode([description]).astype('float32')
distances, indices = index.search(query_vector, k=10)

In [ ]:
retrieved_candidates = []
for i in range(indices.shape[1]):
    idx = indices[0][i]
    retrieved_candidates.append(vocab_df.iloc[idx])

for i, candidate in enumerate(retrieved_candidates):
    print(f"{i+1}. Label: {candidate['term']}, vocab: {candidate['source_vocabulary']}")
    # if a definition is not Nan, print it just below and indented
    if candidate['definition'] and not pd.isna(candidate['definition']):
        print(f"    Definition: {candidate['definition']}")

### Example of RAG classification

In [ ]:
import os
from classification import RAGObjectClassifier
from classification import ObjectCategory, ObjectCategoriser

In [ ]:
GEMINI_API_KEY = ""

In [ ]:
classifier = RAGObjectClassifier(
    index_path="vocabularies/vectordb/all_concepts_min.index",
    metadata_path="vocabularies/vectordb/metadata.pkl",
    generator_model='gemini-2.5-flash',
    embedding_model='all-MiniLM-L6-v2',
    generator_api_key=GEMINI_API_KEY,    
)

In [ ]:
vocab_number, vocab_candidates = classifier.classify(
    description="The Blues Brother Gibson ES-335 guitar",
    classification_prompt='single'
)

In [ ]:
vocab_candidates

In [ ]:
print(f"Selected vocabulary number: {vocab_number}: {vocab_candidates[vocab_number[0]]['term']}")

### Example of Object Categorisation

In [ ]:
categoriser = ObjectCategoriser(
    generator_api_key = GEMINI_API_KEY,
    generator_model='gemini-2.5-pro',
)

In [ ]:
categoriser.classify(
    description="The Blues Brother Gibson ES-335 guitar",)

## Running & testing

### Testing Object Categorisation accuracy

In [ ]:
import pandas as pd

item_with_captions = pd.read_csv('../../assets/data/items_with_captions.csv')

In [ ]:
# select only the records with non-null non-empty base_type
test_items = item_with_captions[
    item_with_captions['base_type'].notna() & (item_with_captions['base_type'] != '')]
print(f"Selected {len(test_items)} test items.")

In [ ]:
# isolate only the item_description and the base_type columns
test_items = test_items[['item_description', 'base_type']]
# reset the index there
test_items.reset_index(drop=True, inplace=True)
# first we mae sure all the items have a non-empty base_type as assertion
assert (test_items['base_type'] is None or test_items['base_type'].isna()).sum() == 0, \
    "There are records with empty or NaN base_type!"
# there may be items with more than one base_type, separated by a | character,
# therefore we will split them and keep them in a list
test_items['base_type'] = test_items['base_type'].apply(
    lambda x: x.split(' | ') if isinstance(x, str) else [])

# finally we check wether the base_type values are all valid ObjectCategory values
valid_categories = {cat.value for cat in ObjectCategory}
invalid_categories = set()
for types in test_items['base_type']:
    for t in types:
        if t not in valid_categories:
            invalid_categories.add(t)
if invalid_categories:
    raise ValueError(f"Invalid base_type values found: {invalid_categories}")

In [ ]:
test_items

In [ ]:
import time
from tqdm import tqdm

# let's try the classification system on these items via the description, and simply check whether the prediction is in the list of the base_type values
results = []
for idx, row in tqdm(test_items.iterrows(), total=len(test_items), desc="Classifying items"):
    description = row['item_description']
    true_types = row['base_type']
    predicted_type = ObjectCategoriser.classify(description=description)
    # check if any of the predicted types is in the true types
    is_correct = predicted_type in true_types
    results.append({
        'description': description,
        'true_types': true_types,
        'predicted_types': predicted_type,
        'is_correct': is_correct
    })
    # a small delay to avoid hitting rate limits
    time.sleep(0.5)


In [ ]:
# let's measure the accuracy of the categorisation system
results_df = pd.DataFrame(results)
accuracy = results_df['is_correct'].mean()
print(f"Classification accuracy: {accuracy:.2%}")

In [ ]:
# show me the mispredictions
results_df[~results_df['is_correct']]

### Running the RAGObjectClassifier on the testset

#### Preparing the data

In [ ]:
import pandas as pd

item_with_captions = pd.read_csv('../../assets/data/items_with_captions.csv')

In [ ]:
test_items = test_items[test_items["set"] == "test-1"]
print(f"Selected {len(test_items)} test items")

In [ ]:
test_items.columns

In [ ]:
# Since we want to have something complete, we also link the zone/exhibit info
zonemap_df = pd.read_csv('../../assets/data/zone_case_mappings.csv')
zonemap_df.columns

In [ ]:
# Convert types for a clean merge, as before
zonemap_df['case_number'] = zonemap_df['case_number'].astype(int)
zonemap_df['zone'] = zonemap_df['zone'].astype(int)

# Step 1: Select only the 'zone' and 'case_number' columns from zonemap_df
# and rename 'case_number' to 'case' for a clean merge.
merge_df = zonemap_df[['zone', 'case_number']].rename(columns={'case_number': 'case'})

# Step 2: Perform the left merge to add the 'zone' column to test_items.
test_items = pd.merge(test_items, merge_df, on='case', how='left')

# Step 3: Reorder the columns to place 'zone' right after 'case'.
cols = test_items.columns.tolist()
# Remove 'zone' from its current position (at the end)
cols.remove('zone')
# Find the index of 'case'
case_index = cols.index('case')
# Insert 'zone' after 'case'
cols.insert(case_index + 1, 'zone')

# Reassign the DataFrame with the new column order
test_items = test_items[cols]

In [ ]:
test_items = test_items.drop(columns=['set', 'publisher', 'cardinality'])

In [ ]:
test_items.to_csv("../../assets/data/testset_flat_nt.csv", index=None)

#### Loading and testing the classifier

In [1]:
import pandas as pd
from classification import RAGObjectClassifier

testset_df = pd.read_csv('../../assets/data/testset_flat_nt.csv')

/Users/jacopodeberardinis/anaconda3/envs/bmenvai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
testset_df.columns

Index(['item_description', 'case', 'zone', 'base_type', 'specific_type',
       'item_title', 'music_genres', 'music_works', 'music_albums',
       'music_artists', 'events', 'countries', 'places', 'venues', 'years',
       'colours', 'materials', 'caption'],
      dtype='object')

In [3]:
# let's choose an item to start with

sample = testset_df.iloc[29]
sample

item_description    Ziggy Stardust Kimono with Carp Design
case                                                    10
zone                                                     4
base_type                                         Clothing
specific_type                                       Kimono
item_title          Ziggy Stardust Kimono with Carp Design
music_genres                                           NaN
music_works                                            NaN
music_albums                                           NaN
music_artists                                  David Bowie
events                                                 NaN
countries                                              NaN
places                                                 NaN
venues                                                 NaN
years                                                  NaN
colours                                                NaN
materials                                              N

In [5]:
classifier = RAGObjectClassifier(
    index_path="vocabularies/vectordb/all_concepts.index",
    metadata_path="vocabularies/vectordb/metadata.pkl",
    generator_model='gemini-2.5-flash',
    embedding_model='all-MiniLM-L6-v2',
    generator_api_key=GEMINI_API_KEY,    
)


--- Loading FAISS Index from 'vocabularies/vectordb/all_concepts.index' ---
Index loaded. Total entries in index: 50301
--- Loading Metadata from 'vocabularies/vectordb/metadata.pkl' ---
Metadata loaded. Total entries in metadata: 50301
--- Loading SentenceTransformer Model 'all-MiniLM-L6-v2' ---
Model loaded successfully.
LLM Client initialised with model 'gemini-2.5-flash'.


In [6]:
candidates = classifier.retrieve(
    description=sample["item_description"],
    verbose=True
)

Top 10 Retrieved Candidates:
1. Label: kimono sleeve, URI: http://thesaurus.europeanafashion.eu/thesaurus/10846, vocab: Europeana
2. Label: kimonos, URI: http://vocab.getty.edu/aat/300046171, vocab: AAT
3. Label: kimono, URI: http://thesaurus.europeanafashion.eu/thesaurus/10008, vocab: Europeana
4. Label: carp, URI: http://vocab.getty.edu/aat/300250185, vocab: AAT
5. Label: Kimbu, URI: http://vocab.getty.edu/aat/300262876, vocab: AAT
6. Label: kimberlite, URI: http://vocab.getty.edu/aat/300451566, vocab: AAT
7. Label: birchbark artists, URI: http://vocab.getty.edu/aat/300435654, vocab: AAT
8. Label: Raft zither, URI: http://www.mimo-db.eu/InstrumentsKeywords/3688, vocab: MIMO
9. Label: zinnober green, URI: http://vocab.getty.edu/aat/300311406, vocab: AAT
10. Label: zigzags, URI: http://vocab.getty.edu/aat/300165028, vocab: AAT


In [9]:
p = classifier.generate_prompt(
    description=sample["item_description"],
    retrieved_candidates=candidates,
)
print(p)


You are an expert museum cataloguer. Based on the item description below, choose
the single most appropriate specific type from the 'Controlled Vocabulary List'.
Your response must be ONLY the number of the best matching term. If no term is
suitable, respond with 0.

ITEM DESCRIPTION:
Ziggy Stardust Kimono with Carp Design

CONTROLLED VOCABULARY LIST:
(1) Term: kimono sleeve, Definition: , URI: http://thesaurus.europeanafashion.eu/thesaurus/10846
(2) Term: kimonos, Definition: Loose, wide-sleeved garments fastened around the waist with an obi or broad sash, traditionally worn by Japanese men and women., URI: http://vocab.getty.edu/aat/300046171
(3) Term: kimono, Definition: Loose, wide-sleeved garments fastened around the waist with an obi or broad sash, traditionally worn by Japanese men and women (AAT), URI: http://thesaurus.europeanafashion.eu/thesaurus/10008
(4) Term: carp, Definition: General term referring to members of several species of the Cyprinidae family., URI: http://voca

In [12]:
selected_class, candidates  = classifier.classify(
    description=sample["item_description"]
)

In [15]:
for s in selected_class:
    print(candidates[s])

term                                                            kimono
definition           Loose, wide-sleeved garments fastened around t...
uri                  http://thesaurus.europeanafashion.eu/thesaurus...
source_vocabulary                                            Europeana
Name: 48735, dtype: object


### Scaling up with the classifiers 

In [4]:
from classification import VOCAB_MAP, RAGObjectClassifier

aat_classifier = RAGObjectClassifier(
    index_path="vocabularies/aat/vectordb/aat.index",
    metadata_path="vocabularies/aat/vectordb/aat_metadata.pkl",
    generator_model='gemini-2.5-flash',
    embedding_model='all-MiniLM-L6-v2',
    generator_api_key=GEMINI_API_KEY,    
)

mimo_classifier = RAGObjectClassifier(
    index_path="vocabularies/mimo/vectordb/mimo.index",
    metadata_path="vocabularies/mimo/vectordb/mimo_metadata.pkl",
    generator_model='gemini-2.5-flash',
    embedding_model='all-MiniLM-L6-v2',
    generator_api_key=GEMINI_API_KEY,    
)

europeana_classifier = RAGObjectClassifier(
    index_path="vocabularies/fashion/vectordb/europeana.index",
    metadata_path="vocabularies/fashion/vectordb/europeana_metadata.pkl",
    generator_model='gemini-2.5-flash',
    embedding_model='all-MiniLM-L6-v2',
    generator_api_key=GEMINI_API_KEY,
)

pbcore_classifier = RAGObjectClassifier(
    index_path="vocabularies/pbcore/vectordb/pbcore.index",
    metadata_path="vocabularies/pbcore/vectordb/pbcore_metadata.pkl",
    generator_model='gemini-2.5-flash',
    embedding_model='all-MiniLM-L6-v2',
    generator_api_key=GEMINI_API_KEY,    
)


--- Loading FAISS Index from 'vocabularies/aat/vectordb/aat.index' ---
Index loaded. Total entries in index: 46403
--- Loading Metadata from 'vocabularies/aat/vectordb/aat_metadata.pkl' ---
Metadata loaded. Total entries in metadata: 46403
--- Loading SentenceTransformer Model 'all-MiniLM-L6-v2' ---
Model loaded successfully.
LLM Client initialised with model 'gemini-2.5-flash'.

--- Loading FAISS Index from 'vocabularies/mimo/vectordb/mimo.index' ---
Index loaded. Total entries in index: 2324
--- Loading Metadata from 'vocabularies/mimo/vectordb/mimo_metadata.pkl' ---
Metadata loaded. Total entries in metadata: 2324
--- Loading SentenceTransformer Model 'all-MiniLM-L6-v2' ---
Model loaded successfully.
LLM Client initialised with model 'gemini-2.5-flash'.

--- Loading FAISS Index from 'vocabularies/fashion/vectordb/europeana.index' ---
Index loaded. Total entries in index: 1481
--- Loading Metadata from 'vocabularies/fashion/vectordb/europeana_metadata.pkl' ---
Metadata loaded. Total

In [5]:
import enum

classifiers = {
    "AAT": aat_classifier,
    "MIMO": mimo_classifier,
    "Europeana": europeana_classifier,
    "PBCore": pbcore_classifier,
}

VOCAB_MAP

{'Musical Instrument': 'MIMO',
 'Clothing': 'Europeana',
 'Audiovisual Media': 'PBCore',
 'Textual & Graphic Work': 'AAT',
 'Merchandise': 'AAT',
 'Personal Effects': 'AAT',
 'Award & Recognition': 'AAT'}

In [6]:
# remove me later FIXME
candidates = europeana_classifier.retrieve(description="Ziggy Stardust Kimono with Carp Design", verbose=True)

Top 10 Retrieved Candidates:
1. Label: kimono sleeve, URI: http://thesaurus.europeanafashion.eu/thesaurus/10846, vocab: Europeana
2. Label: kimono, URI: http://thesaurus.europeanafashion.eu/thesaurus/10008, vocab: Europeana
3. Label: pagoda sleeve, URI: http://thesaurus.europeanafashion.eu/thesaurus/10847, vocab: Europeana
4. Label: zinc alloy, URI: http://thesaurus.europeanafashion.eu/thesaurus/11421, vocab: Europeana
5. Label: zinc, URI: http://thesaurus.europeanafashion.eu/thesaurus/11130, vocab: Europeana
6. Label: cardigan, URI: http://thesaurus.europeanafashion.eu/thesaurus/10021, vocab: Europeana
7. Label: Zamak (TM), URI: http://thesaurus.europeanafashion.eu/thesaurus/11422, vocab: Europeana
8. Label: batwing sleeve, URI: http://thesaurus.europeanafashion.eu/thesaurus/10844, vocab: Europeana
9. Label: starch, URI: http://thesaurus.europeanafashion.eu/thesaurus/10567, vocab: Europeana
10. Label: metallic, URI: http://thesaurus.europeanafashion.eu/thesaurus/10406, vocab: European

In [ ]:
def classify_object(object_description: int,
                    object_category: enum,
                    classifiers: dict,
                    top_k: int = 10):

    assert object_category in VOCAB_MAP, \
        "The given object category is not supported"
    # Retrieving the vocabulary to use for this object type
    target_vocab = VOCAB_MAP[object_category]
    vocab_classifier = classifiers[target_vocab]
    # Using the category-specific classifier for inference
    selected_class, candidates = vocab_classifier.classify(
        description=object_description,
        top_k=top_k,
    )

    if len(selected_class) == 0:
        print("No class found for the given object")
        return None, candidates
        # TODO Implement fallback mechanism to switch to AAT and try again
        # before giving up and raising a warning if anything is found

    return selected_class, candidates

In [28]:
sel_classes, candidates = classify_object(
    object_description="Fender Stratocaster 1963",
    object_category="Musical Instrument",
    classifiers=classifiers,
)

In [30]:
for s in sel_classes:
    print(candidates[s])

term                                                   Electric guitar
definition           An electric guitar is a guitar that requires e...
uri                     http://www.mimo-db.eu/InstrumentsKeywords/3236
source_vocabulary                                                 MIMO
Name: 47798, dtype: object


In [ ]:
import pandas as pd
from classification import RAGObjectClassifier

testset_df = pd.read_csv('../../assets/data/testset_flat_nt.csv')